In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from keras.models import Sequential
from keras.layers import Dense, Dropout

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

In [2]:
# Load the dataset
df = pd.read_table('C:\\Users\\Dovid Glassner\\Downloads\\CMaps\\train_FD001.txt', sep='\s+', header=None)

# Check if columns 26 and 27 exist in the DataFrame
if 26 in df.columns and 27 in df.columns:
    df.drop([26, 27], axis=1, inplace=True)  # Remove unused columns

df.columns = ['unit', 'cycle', 'op1', 'op2', 'op3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8',
              's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Dovid Glassner\\Downloads\\CMaps\\train_FD001.txt'

In [ ]:
# Data Exploration
print("Dataset Shape: ", df.shape)
print("Columns: ", df.columns)
print("Head of the Dataset:")
print(df.head())

In [ ]:
# Descriptive Statistics
print("Descriptive Statistics:")
print(df.describe())

In [ ]:
# Check for Missing Values
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Visualize the distribution of 'cycle' variable
plt.figure(figsize=(8, 6))
sns.histplot(data=df, x='cycle', kde=True)
plt.xlabel('Cycle')
plt.ylabel('Count')
plt.title('Distribution of Cycles')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(18, 22))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Feature Distribution
fig, axes = plt.subplots(7, 4, figsize=(18, 22))
axes = axes.flatten()
for i, col in enumerate(df.columns[2:]):
    sns.histplot(data=df, x=col, kde=True, ax=axes[i])
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Create the target variable
df['max_cycle'] = df.groupby('unit')['cycle'].transform('max')
df['failed'] = np.where(df['max_cycle'] - df['cycle'] <= 30, 1, 0)
df.drop('max_cycle', axis=1, inplace=True)

In [ ]:
# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(df.drop(['unit', 'cycle', 'failed'], axis=1),
                                                    df['failed'], test_size=0.2, random_state=42)

In [ ]:
# Scale the input features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train logistic regression model
lr = LogisticRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
print('Logistic Regression - Accuracy: {:.3f}, Precision: {:.3f}'.format(accuracy_score(y_test, y_pred_lr),
                                                                          precision_score(y_test, y_pred_lr)))

In [ ]:
# Train decision tree model
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)
y_pred_dt = dt.predict(X_test_scaled)
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
print('Decision Tree - Accuracy: {:.3f}, Precision: {:.3f}'.format(accuracy_score(y_test, y_pred_dt),
                                                                    precision_score(y_test, y_pred_dt)))

In [ ]:
# Train random forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
print('Random Forest - Accuracy: {:.3f}, Precision: {:.3f}'.format(accuracy_score(y_test, y_pred_rf),
                                                                   precision_score(y_test, y_pred_rf)))

In [ ]:
# Train neural network model
model = Sequential()
model.add(Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_data=(X_test_scaled, y_test))
y_pred_nn = np.where(model.predict(X_test_scaled) > 0.5, 1, 0)
nn_accuracy = accuracy_score(y_test, y_pred_nn)
nn_precision = precision_score(y_test, y_pred_nn)
print('Neural Network - Accuracy: {:.3f}, Precision: {:.3f}'.format(accuracy_score(y_test, y_pred_nn),
                                                                    precision_score(y_test, y_pred_nn)))

In [ ]:
# Model Comparison
model_names = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Neural Network']
accuracy_scores = [lr_accuracy, dt_accuracy, rf_accuracy, nn_accuracy]
precision_scores = [lr_precision, dt_precision, rf_precision, nn_precision]

plt.figure(figsize=(10, 6))
sns.barplot(x=model_names, y=accuracy_scores, palette='Blues_r')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.title('Model Comparison - Accuracy')
plt.ylim([0, 1])
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(x=model_names, y=precision_scores, palette='Greens_r')
plt.xlabel('Model')
plt.ylabel('Precision')
plt.title('Model Comparison - Precision')
plt.ylim([0, 1])
plt.show()